# CRIME DNA: Phase 4 — Exploratory Data Analysis & Behavioral Profiling

This notebook provides in-depth exploratory data analysis of the **70-dimensional Behavioral DNA** representations constructed from 220,663 crime incidents.

### Analysis Sections:
1. **Crime Target Distribution**
2. **24-Hour Diurnal & Annual Temporal Patterns**
3. **Nighttime vs Daytime Behavioral Choices**
4. **Weekend vs Weekday Operational Shifts**
5. **Location & Urban Environmental Scale**
6. **Tactical Weapons & Escalation**
7. **Victimology & Offender Group Structure**
8. **Property Targeting & Economic Motivations**
9. **Harm, Narcotics & Incident Outcomes**
10. **Behavioral DNA Correlation Matrix**
11. **Feature Distributions Across Dimensions**
12. **Rare Behavioral Signatures & Anomaly Candidates**

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Configure plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']

# Load Behavioral DNA dataset
dna_path = '../data/processed/crime_dna_behavioral_features.csv'
clean_path = '../data/processed/crime_dna_cleaned.csv'

df_dna = pd.read_csv(dna_path)
df_clean = pd.read_csv(clean_path)
print(f'Behavioral DNA matrix shape: {df_dna.shape}')
print(f'Cleaned source matrix shape: {df_clean.shape}')

## 1. Crime Target Distribution
Examining the core taxonomy: Person, Property, and Society violations.

In [2]:
crime_cats = {
    'Property Only': (df_dna['crime_against_property'] == 1.0) & (df_dna['crime_against_person'] == 0.0) & (df_dna['crime_against_society'] == 0.0),
    'Person Only': (df_dna['crime_against_person'] == 1.0) & (df_dna['crime_against_property'] == 0.0) & (df_dna['crime_against_society'] == 0.0),
    'Society Only': (df_dna['crime_against_society'] == 1.0) & (df_dna['crime_against_person'] == 0.0) & (df_dna['crime_against_property'] == 0.0),
    'Person + Property': df_dna['crime_against_person_property'] == 1.0,
    'Property + Society': df_dna['crime_against_property_society'] == 1.0,
    'Person + Society': df_dna['crime_against_person_society'] == 1.0,
    'Person + Property + Society': df_dna['crime_against_person_property_society'] == 1.0,
}
counts = {k: v.sum() for k, v in crime_cats.items()}
for k, v in counts.items():
    print(f'{k}: {v:,} ({v/len(df_dna)*100:.2f}%)')

## 2. Diurnal and Seasonal Temporal Rhythms
Analyzing how crime peaks during the 24-hour cycle and across calendar months.

In [3]:
hourly = df_clean['incident_hour'].value_counts().sort_index()
print('Peak crime hours:')
print(hourly.sort_values(ascending=False).head(5))

## 3. Behavioral Correlation Analysis
Examining pairwise relationships between key behavioral dimensions.

In [4]:
core_features = [
    'time_is_night', 'time_is_weekend',
    'crime_against_person', 'crime_against_property', 'crime_against_society',
    'has_weapon', 'has_injury', 'has_relationship', 'has_property',
    'has_drug', 'has_arrest_on_scene', 'is_co_offender'
]
corr = df_dna[core_features].corr()
corr.round(3)

## 4. Rare Behavioral Combinations
Detecting candidate anomaly archetypes.

In [5]:
triple = df_dna['crime_against_person_property_society'] == 1.0
print(f'Triple category violations: {triple.sum()} cases ({triple.sum()/len(df_dna)*100:.3f}%)')

cargo_armed = (df_dna['is_cargo_theft'] == 1.0) & (df_dna['has_weapon'] == 1.0)
print(f'Armed freight cargo theft: {cargo_armed.sum()} cases ({cargo_armed.sum()/len(df_dna)*100:.3f}%)')